# Retrieval Encoder Comparison

Compares candidate image encoders for the V-RAG retrieval step and measures which one retrieves
the most clinically relevant neighbours. The winning encoder is then used to build the retrieval
database in the next notebook.

## Inputs
- `chexpert_cache/` from the previous notebook (image shards + manifest + report table)
- A Redivis API token, used only to pull the finding labels that score retrieval quality
- `test_vqa.jsonl`, whose patients are excluded so the evaluation benchmark stays held out

## Outputs
- One retrieval database per encoder: `vrag_db_<encoder>/` (embeddings and FAISS index)
- A head-to-head table reporting retrieval quality for each encoder

## Encoders compared
- BiomedCLIP - a general biomedical image-text model
- CheXzero - a contrastive model trained specifically on chest X-rays

## Role in the pipeline
Retrieval quality is the ceiling on what V-RAG can achieve: if the retrieved neighbours are not
clinically similar to the query, the references handed to the model are noise. This notebook
selects the encoder on measured retrieval lift rather than assumption.

## Configuration

Defines which encoders to compare, the retrieval depth, the size of the held-out query pool, and
the label conventions. Uncertain findings are treated as present, which matches how the
evaluation benchmark was labelled, so retrieval scores are measured on the same convention.

In [ ]:
# ============================================================
#  CONFIG
# ============================================================
REDIVIS_TOKEN = "AAAGwGJS33f4OxyGTuwmWWyxRfuivWks"   # labels only — no images from Redivis
DATASET_REF   = "chexpert_plus:5yyj"
LABELS_TABLE  = "chexpert_labels:pmec"               # report_fixed.json lives here
LABELS_FILE   = "report_fixed.json"                  # labels the FULL report (matches LLaVA_S)

TEST_VQA_PATH = "/kaggle/input/datasets/1000-test-img-id-vqa-/vqa/test_vqa.jsonl"

ONE_PER          = "patient"
N_QUERY_PATIENTS = 13000
TOP_K            = 3
SEED             = 42
IMG_SIZE         = 336        # the cache size; each encoder resizes down from here
BATCH            = 256
WORKERS          = 4

# uncertain (-1.0) counts as POSITIVE (U-Ones) — matches how test_vqa was built
UNCERTAIN_IS_POSITIVE = True

# which encoders to attempt. Removing one just skips it — order is the run order.
ENCODERS = ["biomedclip", "chexzero"]

OUT_ROOT = "/working"     # each db -> OUT_ROOT/vrag_db_<name>/

import os, json, random
from pathlib import Path
random.seed(SEED)
os.environ["REDIVIS_API_TOKEN"] = load  token
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["REDIVIS_API_TOKEN"] = UserSecretsClient().get_secret("REDIVIS_API_TOKEN") or REDIVIS_TOKEN
except Exception:
    pass
print("encoders to build:", ENCODERS)

## Install

Installs the encoder libraries and FAISS, which provides the similarity index used for retrieval.

In [ ]:
# open_clip -> BiomedCLIP ; faiss -> index ; gdown -> CheXzero ckpt ; CLIP -> CheXzero arch
!pip install -q open_clip_torch faiss-cpu gdown redivis
!pip install -q git+https://github.com/openai/CLIP.git
import faiss, torch, numpy as np
print(f"faiss {faiss.__version__} | torch {torch.__version__} | "
      f"cuda {torch.cuda.is_available()} {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '— NO GPU'}")
device = "cuda" if torch.cuda.is_available() else "cpu"

## Locate the cached images and report table

Finds the mounted cache and builds an index from image key to the shard containing it, so images
can be read directly out of the compressed shards.

In [ ]:
# ============================================================
#  Cell 2a — Cached images + reports (from the mounted dataset)
# ============================================================
import pandas as pd, zipfile, io, csv, time

# Locate the cache folder (the one that holds images_part* and the parquet).
CACHE_HINT = "input/datasets/cached-data-chexpert/chexpert_cache"
cands = [Path(CACHE_HINT), Path("/input/cached-data-chexpert/chexpert_cache")] + \
        [p.parent for p in Path(/input").rglob("chexpert_plus_full.parquet")]
CACHE = next((c for c in cands if c.exists() and list(c.glob("images_part*"))), None)
if CACHE is None:
    print("dirs under /input:")
    for d in Path("/input").rglob("*"):
        if d.is_dir(): print("  ", d)
    raise SystemExit("cache not found — check Add Data / the path")
print(f"cache: {CACHE}")

# images_part* is EITHER zip files (as built) OR directories — Kaggle auto-extracts zips
# on dataset upload, so images_part00.zip becomes images_part00/patient.../....png.
# Handle both. key -> ("zip", shard_path) | ("file", png_path)   (key = patient.../view.png)
key2src, n_zip, n_dir = {}, 0, 0
for p in sorted(CACHE.glob("images_part*")):
    if p.is_dir():
        n_dir += 1
        for png in p.rglob("*.png"):
            key2src[str(png.relative_to(p))] = ("file", str(png))
    elif zipfile.is_zipfile(p):
        n_zip += 1
        with zipfile.ZipFile(p) as zf:
            for n in zf.namelist():
                if n.endswith(".png"): key2src[n] = ("zip", str(p))
print(f"   images_part*: {n_zip} zip + {n_dir} dir  ->  {len(key2src):,} image keys")
if not key2src:
    raise SystemExit("no .png found inside images_part* (neither zip members nor loose files)")
# show the shape of a key so we can confirm it matches to_png_key()'s output
print(f"   sample key: {next(iter(key2src))}")

# reports (text) from the cached parquet
pq = list(CACHE.rglob("chexpert_plus_full.parquet"))
if not pq:
    raise SystemExit("chexpert_plus_full.parquet not in the cache mount")
raw_df = pd.read_parquet(pq[0], columns=["path_to_image","deid_patient_id",
                                         "patient_report_date_order","frontal_lateral","split","report"])
print(f"   reports parquet: {len(raw_df):,} rows")

## Download the finding labels

Pulls the per-report finding labels from Redivis. These labels are used only to score how
clinically relevant the retrieved neighbours are; they play no part in retrieval itself, which
is purely image-embedding based.

In [ ]:
# ============================================================
#  Cell 2b — Labels (report_fixed) from Redivis  — the only Redivis touch
# ============================================================
import io, json, redivis
import redivis.common.TabularReader as _TR
if not getattr(_TR, "_sr", False):
    _o = _TR.make_request
    def _b(*a, **k):
        r = _o(*a, **k); p = k.get("path","") or ""
        if isinstance(p,str) and p.endswith("/rawFiles") and getattr(r,"status_code",None)==200:
            r.raw = io.BytesIO(r.raw.read(decode_content=True))
        return r
    _TR.make_request = _b; _TR._sr = True

ds = redivis.organization("AIMI").dataset(DATASET_REF); ds.get()
lt = ds.table(LABELS_TABLE); lt.get()
print("pulling labels from Redivis ...")
blob = next(f for f in lt.list_files() if str(f.path) == LABELS_FILE).read()

CHEXPERT_14 = ["Enlarged Cardiomediastinum","Cardiomegaly","Lung Opacity","Lung Lesion","Edema",
               "Consolidation","Pneumonia","Atelectasis","Pneumothorax","Pleural Effusion",
               "Pleural Other","Fracture","Support Devices","No Finding"]
POS = {1.0, -1.0} if UNCERTAIN_IS_POSITIVE else {1.0}

def to_png_key(p):
    parts = list(Path(str(p).strip()).parts)
    while parts and (parts[0] in ("train","valid","test") or parts[0].startswith("CheXpert")):
        parts = parts[1:]
    return str(Path(*parts).with_suffix(".png"))

labels = {}          # png_key -> frozenset(present entities, excluding No Finding)
for line in io.BytesIO(blob).read().decode().splitlines():
    if not line.strip(): continue
    r = json.loads(line)
    labels[to_png_key(r["path_to_image"])] = frozenset(
        e for e in CHEXPERT_14 if e != "No Finding" and r.get(e) in POS)
print(f"labels: {len(labels):,}  (present = {{1.0,-1.0}} -> U-Ones)")

## Build one shared evaluation split

Reduces the pool to one frontal image per patient from the training split, excludes the
benchmark patients, and carves a fixed query pool out of the remaining memory pool. The same
split is reused for every encoder so the comparison is fair.

In [ ]:
# ============================================================
#  One split, shared by every encoder -> fair comparison
# ============================================================
import numpy as np
df = raw_df.copy()
df = df[df["frontal_lateral"].str.lower().str.strip() == "frontal"]
df = df[df["split"].str.lower().str.strip() == "train"]
df["report"] = df["report"].fillna("").astype(str).str.strip()
df = df[df["report"].str.len() > 0]
df["png_key"] = df["path_to_image"].map(to_png_key)
df["study"]   = df["png_key"].map(lambda k: "/".join(k.split("/")[:2]))
df = df[df["png_key"].isin(key2zip)]                      # image must be in the cache

sc = [c for c in ["deid_patient_id","patient_report_date_order","png_key"] if c in df.columns]
df = df.sort_values(sc, kind="mergesort")
entries = df.drop_duplicates(subset=("deid_patient_id" if ONE_PER=="patient" else "study"),
                             keep="first").reset_index(drop=True)

# test patients out (resolve the mounted path)
tv = TEST_VQA_PATH
if tv and not Path(tv).exists():
    hits = sorted(Path("/input").rglob(Path(tv).name)); tv = str(hits[0]) if hits else None
test_patients = set()
if tv:
    for l in open(tv):
        if not l.strip(): continue
        r = json.loads(l)
        p = r.get("deid_patient_id") or r.get("patient_id") or r.get("patient")
        if not p:
            for f_ in ("image","image_path","path_to_image","png_key","path","image_id"):
                if r.get(f_):
                    c = to_png_key(r[f_]).split("/")[0]
                    if c.startswith("patient"): p = c; break
        if p: test_patients.add(str(p))
    if not test_patients: raise SystemExit("test_vqa parsed but no patient IDs — check schema")
    print(f"test patients excluded: {len(test_patients):,}")
else:
    print("no test_vqa — test patients NOT excluded")

entries = entries[~entries["deid_patient_id"].astype(str).isin(test_patients)]
allp = sorted(entries["deid_patient_id"].astype(str).unique())
rng = np.random.RandomState(SEED); rng.shuffle(allp)
nq = min(N_QUERY_PATIENTS, len(allp)//4)
qp_, mp_ = set(allp[:nq]), set(allp[nq:])
memory_df = entries[entries["deid_patient_id"].astype(str).isin(mp_)].reset_index(drop=True)
query_df  = entries[entries["deid_patient_id"].astype(str).isin(qp_)].reset_index(drop=True)

mp = set(memory_df["deid_patient_id"].astype(str)); qp = set(query_df["deid_patient_id"].astype(str))
assert not (mp & qp) and not (mp & test_patients) and not (qp & test_patients)
assert not (set(memory_df["png_key"]) & set(query_df["png_key"]))

# label coverage on the split (used by scoring)
mem_lab = [labels.get(k, frozenset()) for k in memory_df["png_key"]]
qry_lab = [labels.get(k, frozenset()) for k in query_df["png_key"]]
print(f"M {len(memory_df):,} | queries {len(query_df):,} | patient-disjoint")
print(f"   label coverage: memory {sum(1 for s in mem_lab if s)/len(mem_lab)*100:.0f}% "
      f"| queries {sum(1 for s in qry_lab if s)/len(qry_lab)*100:.0f}% have >=1 positive")

## Encoder registry

Defines how each encoder is loaded, including its own preprocessing and embedding function.
Each loader is isolated so a failure to load one encoder skips it rather than aborting the run.
Note that CheXzero requires non-standard preprocessing (0-255 scale with chest-X-ray-specific
normalisation), which is why preprocessing is attached per encoder rather than shared.

In [ ]:
# ============================================================
#  Three encoders. Each isolated; a failure to load skips it.
# ============================================================
import torch, numpy as np
from torchvision.transforms import Resize, Normalize, InterpolationMode

def load_biomedclip():
    import open_clip
    m, _, pre = open_clip.create_model_and_transforms(
        "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    m = m.to(device).eval()
    dim = m.visual.output_dim if hasattr(m.visual, "output_dim") else 512
    def tf(pil): return pre(pil.convert("RGB"))
    def enc(model, b): return model.encode_image(b)
    return m, tf, enc, dim, 224

def load_chexzero():
    # checkpoint on Google Drive; architecture inferred from the state dict via OpenAI CLIP
    import gdown, clip, glob
    dst = "/working/_chexzero"; os.makedirs(dst, exist_ok=True)
    if not glob.glob(dst + "/**/*.pt", recursive=True):
        gdown.download_folder(
            "https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno",
            output=dst, quiet=True, use_cookies=False)
    ck = sorted(glob.glob(dst + "/**/*.pt", recursive=True), key=os.path.getsize)
    if not ck: raise RuntimeError("no .pt checkpoint downloaded from Drive")
    sd = torch.load(ck[0], map_location="cpu")
    sd = sd.get("state_dict", sd)
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    m = clip.model.build_model(sd).to(device).eval()      # infers arch (dim, patch, res)
    res = m.visual.input_resolution
    dim = m.text_projection.shape[1]
    # CheXzero preprocessing: 0-255 scale (NO /255), CXR-specific mean/std
    rz = Resize(res, interpolation=InterpolationMode.BICUBIC, antialias=True)
    nm = Normalize([101.48761]*3, [83.43944]*3)
    def tf(pil):
        a = np.asarray(pil.convert("RGB")).astype(np.float32)     # 0-255
        return nm(rz(torch.from_numpy(a).permute(2, 0, 1)))
    def enc(model, b): return model.encode_image(b)
    return m, tf, enc, dim, res

LOADERS = {"biomedclip": load_biomedclip, "chexzero": load_chexzero}
print("registry:", list(LOADERS))

## Embed and index each encoder

For every encoder: embeds the whole pool, builds a FAISS index over the memory set, retrieves
the top-k neighbours for each query, and scores how often those neighbours share the query's
findings. Retrieval lift is reported against a random-neighbour baseline, which isolates genuine
retrieval quality from the base rate of the findings.

In [ ]:
import zipfile, io, time, traceback, json
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

class CacheDS(Dataset):
    """Reads PNGs straight from the ZIP_STORED shards, fork-safe (per-process handle)."""
    def __init__(self, keys, tf): self.keys=list(keys); self.tf=tf; self._h={}; self._pid=None
    def _zf(self, p):
        if self._pid != os.getpid(): self._h, self._pid = {}, os.getpid()
        if p not in self._h: self._h[p] = zipfile.ZipFile(p)
        return self._h[p]
    def __len__(self): return len(self.keys)
    def __getitem__(self, i):
        k = self.keys[i]
        img = Image.open(io.BytesIO(self._zf(key2zip[k]).read(k)))
        return self.tf(img), i

@torch.no_grad()
def embed_all(keys, tf, enc, model, dim, tag):
    out = np.zeros((len(keys), dim), dtype=np.float32)
    dl = DataLoader(CacheDS(keys, tf), batch_size=BATCH, num_workers=WORKERS,
                    pin_memory=(device=="cuda"))
    for px, idx in tqdm(dl, desc=tag, unit="b"):
        f = enc(model, px.to(device, non_blocking=True))
        f = torch.nn.functional.normalize(f.float(), dim=-1)
        out[idx.numpy()] = f.cpu().numpy()
    return out

@torch.no_grad()
def verify_gate(tf, enc, model, dim):
    # same image twice -> ~1.0 ; a handful of images -> real spread, no NaN
    sample = memory_df["png_key"].iloc[:64].tolist()
    E = embed_all(sample, tf, enc, model, dim, "verify")
    if not np.isfinite(E).all():                       raise RuntimeError("NaN/inf embeddings")
    one = float((E[0] * embed_all([sample[0]], tf, enc, model, dim, "verify1")[0]).sum())
    if one < 0.999:                                    raise RuntimeError(f"same-image sim {one:.3f} != 1")
    S = E @ E.T
    offdiag = S[~np.eye(len(E), dtype=bool)]           # true off-diagonal only — a zeroed
    if offdiag.std() < 1e-3:                            # diagonal would fake variance and let
        raise RuntimeError("degenerate: all embeddings identical")  # identical embeddings pass
    print(f"   gate same-image={one:.4f}  off-diag sim: mean {offdiag.mean():.3f} std {offdiag.std():.3f}")

def jac(a, b):
    u = len(a | b); return len(a & b) / u if u else 1.0

def score(ids, sims, tag):
    top1 = sims[:, 0]
    dup = int((top1 > 0.999).sum())
    # real-label lift, retrieved vs random
    ret = np.mean([np.mean([jac(qry_lab[i], mem_lab[j]) for j in ids[i]]) for i in range(len(ids))])
    rnd_idx = np.random.RandomState(SEED).randint(0, len(mem_lab), (len(ids), TOP_K))
    rnd = np.mean([np.mean([jac(qry_lab[i], mem_lab[j]) for j in rnd_idx[i]]) for i in range(len(ids))])
    lift = (ret - rnd) / rnd * 100 if rnd else float("nan")
    # per-entity: of queries positive for e, what frac of top-K are also positive for e
    per = {}
    for e in CHEXPERT_14:
        if e == "No Finding": continue
        qi = [i for i in range(len(ids)) if e in qry_lab[i]]
        if len(qi) < 20: continue
        r = np.mean([np.mean([e in mem_lab[j] for j in ids[i]]) for i in qi])
        b = float(np.mean([e in s for s in mem_lab]))       # base rate
        per[e] = {"n": len(qi), "retrieved": round(float(r),3), "base": round(b,3),
                  "lift": round((r-b)/b*100,1) if b else None}
    print(f"   top1 sim median {np.median(top1):.3f} | dup {dup} | "
          f"REAL-LABEL lift {lift:+.1f}%  (ret {ret:.3f} vs rnd {rnd:.3f})")
    return {"dup": dup, "lift_pct": round(lift,2), "ret": round(float(ret),4),
            "rnd": round(float(rnd),4), "top1_median": round(float(np.median(top1)),4),
            "per_entity": per}

RESULTS = {}
for name in ENCODERS:
    print("\n" + "="*64); print(f"  ENCODER: {name}"); print("="*64)
    t0 = time.time()
    try:
        model, tf, enc, dim, res = LOADERS[name]()
        print(f"   loaded: dim={dim} res={res}")
        verify_gate(tf, enc, model, dim)

        mem_emb = embed_all(memory_df["png_key"].tolist(), tf, enc, model, dim, f"{name}:memory")
        qry_emb = embed_all(query_df["png_key"].tolist(),  tf, enc, model, dim, f"{name}:queries")
        for nm_, e in (("memory",mem_emb),("queries",qry_emb)):
            assert np.isfinite(e).all() and (np.abs(e).sum(1) > 0).all(), f"bad {nm_} embeddings"

        index = faiss.IndexFlatIP(dim); index.add(mem_emb)
        sims, ids = index.search(qry_emb, TOP_K)
        RESULTS[name] = score(ids, sims, name)

        # ---- save this database independently, right now ----
        OUT = Path(OUT_ROOT) / f"vrag_db_{name}"; OUT.mkdir(parents=True, exist_ok=True)
        faiss.write_index(index, str(OUT/"vrag_index.faiss"))
        keep = ["png_key","deid_patient_id","study","report"]
        memory_df[keep].to_parquet(OUT/"vrag_memory.parquet", index=False)
        query_df[keep].to_parquet(OUT/"vrag_queries.parquet", index=False)
        np.save(OUT/"vrag_memory_emb.npy", mem_emb); np.save(OUT/"vrag_queries_emb.npy", qry_emb)
        with open(OUT/"vrag_retrievals.jsonl","w") as f:
            for i in range(len(query_df)):
                q = query_df.iloc[i]
                f.write(json.dumps({"query_png_key":q["png_key"],"query_patient":str(q["deid_patient_id"]),
                    "query_report":q["report"],
                    "retrieved":[{"png_key":memory_df.iloc[int(j)]["png_key"],
                                  "report":memory_df.iloc[int(j)]["report"],"sim":float(s)}
                                 for j,s in zip(ids[i],sims[i])]})+"\n")
        json.dump({"encoder":name,"embed_dim":int(dim),"resolution":int(res),
                   "memory_entries":len(memory_df),"query_entries":len(query_df),
                   "test_patients_excluded":len(test_patients),"seed":SEED,
                   **RESULTS[name]}, open(OUT/"vrag_manifest.json","w"), indent=2)
        # zip for the RunPod pull
        import zipfile as zf_
        with zf_.ZipFile(f"{OUT_ROOT}/vrag_db_{name}.zip","w",zf_.ZIP_DEFLATED,allowZip64=True) as z:
            for p in sorted(OUT.iterdir()): z.write(p, arcname=f"vrag_db_{name}/{p.name}")
        del model, mem_emb, qry_emb, index
        if device=="cuda": torch.cuda.empty_cache()
        print(f"   {name} done & saved in {(time.time()-t0)/60:.0f} min")
    except Exception as e:
        RESULTS[name] = {"error": f"{type(e).__name__}: {e}"}
        print(f"   {name} SKIPPED — {type(e).__name__}: {e}")
        print("   " + "\n   ".join(traceback.format_exc().splitlines()[-4:]))
        if device=="cuda":
            try: torch.cuda.empty_cache()
            except Exception: pass

## Head-to-head comparison

Prints the final comparison table: embedding dimension, retrieval lift over random, and top-1
similarity per encoder. The encoder with the highest lift is the one carried forward.

In [ ]:
# ============================================================
#  Head-to-head — the table for the paper
# ============================================================
ok = {k: v for k, v in RESULTS.items() if "error" not in v}
print("="*64)
print(f"{'encoder':<12}{'dim':>5}{'lift %':>10}{'ret':>8}{'rnd':>8}{'top1':>8}")
print("-"*64)
for k, v in sorted(ok.items(), key=lambda x: -x[1]["lift_pct"]):
    d = json.load(open(f"{OUT_ROOT}/vrag_db_{k}/vrag_manifest.json"))["embed_dim"]
    print(f"{k:<12}{d:>5}{v['lift_pct']:>9.1f}%{v['ret']:>8.3f}{v['rnd']:>8.3f}{v['top1_median']:>8.3f}")
for k, v in RESULTS.items():
    if "error" in v: print(f"{k:<12}  SKIPPED: {v['error']}")
print("="*64)

if ok:
    best = max(ok, key=lambda k: ok[k]["lift_pct"])
    print(f"\n best real-label lift: {best}  (+{ok[best]['lift_pct']:.1f}%)")
    print("\nper-entity retrieval lift vs base rate (higher = finds that finding better):")
    ents = sorted({e for v in ok.values() for e in v["per_entity"]})
    hdr = "  ".join(f"{k[:9]:>9}" for k in ok)
    print(f"  {'entity':<24}{hdr}")
    for e in ents:
        row = "  ".join(f"{ok[k]['per_entity'].get(e,{}).get('lift','—')!s:>9}" for k in ok)
        print(f"  {e:<24}{row}")

json.dump(RESULTS, open(f"{OUT_ROOT}/encoder_comparison.json","w"), indent=2)
print(f"\nsaved encoder_comparison.json")
print("outputs:")
for p in sorted(Path(OUT_ROOT).glob("vrag_db_*.zip")):
    print(f"   {p.name:28s} {p.stat().st_size/1e6:7.0f} MB")

## Using the result downstream

The encoder with the highest retrieval lift is used to build the production retrieval database
in the V-RAG dataset builder. In this project CheXzero was selected, and its preprocessing is
reproduced exactly in every later notebook so that query and database embeddings share one
vector space.